# Lecture 15

## Pandas: Summaries with Pivot Tables and Group by

## Week 5 Friday

## Miles Chen, PhD

Adapted from Python Data Science Handbook by Jake VanderPlas and Python for Data Analysis by Wes McKinney

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.__version__

'3.0.1'

# Some important data transformation tools

## Multi Index, Hierarchical Indexing

In [3]:
# manual multi-index creation:
my_index = pd.MultiIndex.from_arrays(
    [
    ['a','a','a','b','b','b','c','c','c'],
    [ 4 , 5 , 6 , 4 , 5 , 6 , 4 , 5 , 6 ]
])

data = pd.Series([4, 5, 6, 8, 10, 12, 12, 15, 18], index=my_index)

In [4]:
data

a  4     4
   5     5
   6     6
b  4     8
   5    10
   6    12
c  4    12
   5    15
   6    18
dtype: int64

In [5]:
data.index

MultiIndex([('a', 4),
            ('a', 5),
            ('a', 6),
            ('b', 4),
            ('b', 5),
            ('b', 6),
            ('c', 4),
            ('c', 5),
            ('c', 6)],
           )

In [6]:
# select via the outer index
data.loc['b']

4     8
5    10
6    12
dtype: int64

In [7]:
# select via the inner index
data.loc[:,5] 

a     5
b    10
c    15
dtype: int64

In [8]:
type(data.loc[:,5])

pandas.Series

In [9]:
data.loc[:,5].index

Index(['a', 'b', 'c'], dtype='str')

In [10]:
# the unstack function returns a new DataFrame where the values have been unstacked
# similar to tidyr's spread()/pivot_wider function in R
data.unstack()

,4,5,6
a,4,5,6
b,8,10,12
c,12,15,18


In [11]:
# after unstacking, the index is no longer a multi index
data.unstack().index

Index(['a', 'b', 'c'], dtype='str')

In [12]:
data.unstack().shape

(3, 3)

In [13]:
# the inverse operation of unstack() is stack()
# applying both of these functions will return the same series
data.unstack().stack()

a  4     4
   5     5
   6     6
b  4     8
   5    10
   6    12
c  4    12
   5    15
   6    18
dtype: int64

In [14]:
# you can swap the levels of the multi index using swaplevel
data.swaplevel()

4  a     4
5  a     5
6  a     6
4  b     8
5  b    10
6  b    12
4  c    12
5  c    15
6  c    18
dtype: int64

In [15]:
# the .loc accessors work as expected
data.swaplevel().loc[:,'a']

4    4
5    5
6    6
dtype: int64

In [16]:
# swaplevel will keep the original order
# you may want to sort based on the new swapped index levels
# you must save the output as data remains unchanged
data.swaplevel().sort_index()

4  a     4
   b     8
   c    12
5  a     5
   b    10
   c    15
6  a     6
   b    12
   c    18
dtype: int64

In [17]:
print(data)

a  4     4
   5     5
   6     6
b  4     8
   5    10
   6    12
c  4    12
   5    15
   6    18
dtype: int64


In [18]:
data.swaplevel().unstack()

,a,b,c
4,4,8,12
5,5,10,15
6,6,12,18


In [19]:
# compare to:
data.unstack()

,4,5,6
a,4,5,6
b,8,10,12
c,12,15,18


In [20]:
# summing and other aggregate functions can be performed on an index-based level
# calling sum() on a series, will sum the whole series
data.sum()

np.int64(90)

In [21]:
# you can call groupby on level 0 (the first level of the index) and then sum
# we get sums for each value in the first level of the index
# we will cover groupby in more detail later
data.groupby(level = 0).sum()

a    15
b    30
c    45
dtype: int64

In [22]:
data.groupby(level = 1).sum()

4    24
5    30
6    36
dtype: int64

# Reshaping and Pivoting Data

In [23]:
data = pd.DataFrame(np.arange(1,7).reshape((2, 3)),
                    index  = pd.Index(['alpha', 'beta'], name='letter'),
                    columns= pd.Index(['one', 'two', 'three'], name = 'number'))
data

number,one,two,three
letter,,,
alpha,1,2,3
beta,4,5,6


In [24]:
data.stack()  # creates a multi-index

letter  number
alpha   one       1
        two       2
        three     3
beta    one       4
        two       5
        three     6
dtype: int64

In [25]:
data.stack().unstack()  # unstack undoes the creation of the stacks

number,one,two,three
letter,,,
alpha,1,2,3
beta,4,5,6


In [26]:
data.stack().unstack(0) # you can specify how the unstacking should be done
# here we specify that we should unstack the first level of the multi-index

letter,alpha,beta
number,,
one,1,4
two,2,5
three,3,6


In [27]:
data.stack().unstack('letter')
# you can specify the unstacking by the index level name

letter,alpha,beta
number,,
one,1,4
two,2,5
three,3,6


In [28]:
data.stack().unstack('number')

number,one,two,three
letter,,,
alpha,1,2,3
beta,4,5,6


### Unstacking can introduce missing values

In [29]:
s1 = pd.Series([0, 1, 2, 3], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([4, 5, 6], index=['c', 'd', 'e'])
data2 = pd.concat([s1, s2], keys=['one', 'two'])  
# using the argument keys when concat series will produce a multi-index
data2

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: int64

In [30]:
data2.unstack()

,a,b,c,d,e
one,0.0,1.0,2.0,3.0,NaN
two,NaN,NaN,4.0,5.0,6.0


In [31]:
data2.unstack().stack() # stack() will filter out missing values

one  a    0.0
     b    1.0
     c    2.0
     d    3.0
     e    NaN
two  a    NaN
     b    NaN
     c    4.0
     d    5.0
     e    6.0
dtype: float64

# Small example data wrangling

In [32]:
data = pd.read_csv('../data/macrodata.csv')

https://www.statsmodels.org/dev/datasets/generated/macrodata.html

In [33]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   year      203 non-null    float64
 1   quarter   203 non-null    float64
 2   realgdp   203 non-null    float64
 3   realcons  203 non-null    float64
 4   realinv   203 non-null    float64
 5   realgovt  203 non-null    float64
 6   realdpi   203 non-null    float64
 7   cpi       203 non-null    float64
 8   m1        203 non-null    float64
 9   tbilrate  203 non-null    float64
 10  unemp     203 non-null    float64
 11  pop       203 non-null    float64
 12  infl      203 non-null    float64
 13  realint   203 non-null    float64
dtypes: float64(14)
memory usage: 22.3 KB


In [34]:
data.head()

,year,quarter,realgdp,realcons,realinv,realgovt,realdpi,cpi,m1,tbilrate,unemp,pop,infl,realint
0,1959.0,1.0,2710.349,1707.4,286.898,470.045,1886.9,28.98,139.7,2.82,5.8,177.146,0.00,0.00
1,1959.0,2.0,2778.801,1733.7,310.859,481.301,1919.7,29.15,141.7,3.08,5.1,177.830,2.34,0.74
2,1959.0,3.0,2775.488,1751.8,289.226,491.260,1916.4,29.35,140.5,3.82,5.3,178.657,2.74,1.09
3,1959.0,4.0,2785.204,1753.7,299.356,484.052,1931.3,29.37,140.0,4.33,5.6,179.386,0.27,4.06
4,1960.0,1.0,2847.699,1770.5,331.722,462.199,1955.5,29.54,139.6,3.50,5.2,180.007,2.31,1.19


https://pandas.pydata.org/pandas-docs/stable/generated/pandas.PeriodIndex.html

In [35]:
# We can create a time based index of periods consisting of the year and quarter
periods = pd.PeriodIndex.from_fields(year = data.year, quarter = data.quarter)

In [36]:
periods

PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', length=203)

In [37]:
columns = pd.Index(['realgdp', 'infl', 'unemp'], name = 'item')
columns

Index(['realgdp', 'infl', 'unemp'], dtype='str', name='item')

In [38]:
data = data.reindex(columns = columns) # forces columns to conform to the column index we specified

In [39]:
data.head(10)

item,realgdp,infl,unemp
0,2710.349,0.00,5.8
1,2778.801,2.34,5.1
2,2775.488,2.74,5.3
3,2785.204,0.27,5.6
4,2847.699,2.31,5.2
5,2834.390,0.14,5.2
6,2839.022,2.70,5.6
7,2802.616,1.21,6.3
8,2819.264,-0.40,6.8
9,2872.005,1.47,7.0


In [40]:
periods.to_timestamp('D','start')  # changes 1959Q1 to a date: the start date of Q1 of 1959: 1959-01-01

DatetimeIndex(['1959-01-01', '1959-04-01', '1959-07-01', '1959-10-01',
               '1960-01-01', '1960-04-01', '1960-07-01', '1960-10-01',
               '1961-01-01', '1961-04-01',
               ...
               '2007-04-01', '2007-07-01', '2007-10-01', '2008-01-01',
               '2008-04-01', '2008-07-01', '2008-10-01', '2009-01-01',
               '2009-04-01', '2009-07-01'],
              dtype='datetime64[us]', length=203, freq='QS-OCT')

In [41]:
# the current index is just integers, and we want to replace it
data.index

RangeIndex(start=0, stop=203, step=1)

In [42]:
# specify a new index directly
data.index = periods.to_timestamp('D','start')

In [43]:
data.head()

item,realgdp,infl,unemp
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


In [44]:
data.reset_index() # resets the index to the default integer index and moves the current index into a 
# column named 'index'

item,index,realgdp,infl,unemp
0,1959-01-01,2710.349,0.00,5.8
1,1959-04-01,2778.801,2.34,5.1
2,1959-07-01,2775.488,2.74,5.3
3,1959-10-01,2785.204,0.27,5.6
4,1960-01-01,2847.699,2.31,5.2
...,...,...,...,...
198,2008-07-01,13324.600,-3.16,6.0
199,2008-10-01,13141.920,-8.79,6.9
200,2009-01-01,12925.410,0.94,8.1
201,2009-04-01,12901.504,3.37,9.2


In [45]:
# pandas has a melt function that is similar to tidyr's pivot_longer() function in R
# It takes values in multiple columns and stacks them into a single column, 
# while keeping the other columns as identifier variables

ldata = data.reset_index().melt(
    id_vars='index', 
    var_name='item', 
    value_name='value'
)

In [46]:

ldata = ldata.rename(columns={'index': 'date'})

In [47]:
ldata

,date,item,value
0,1959-01-01,realgdp,2710.349
1,1959-04-01,realgdp,2778.801
2,1959-07-01,realgdp,2775.488
3,1959-10-01,realgdp,2785.204
4,1960-01-01,realgdp,2847.699
...,...,...,...
604,2008-07-01,unemp,6.000
605,2008-10-01,unemp,6.900
606,2009-01-01,unemp,8.100
607,2009-04-01,unemp,9.200


In [48]:
# the shape of ldata is 609 by 3
# there are 609 rows - each date has three rows
# the three columns are date, item name, value
ldata.shape

(609, 3)

In [49]:
# unstack doesn't work, because the stacking and unstacking is powered by multi-index
# instead, all the dates, item names, and values get 'flowed' into one column
# Notice the length is now 1827
ldata.unstack()

date   0      1959-01-01 00:00:00
       1      1959-04-01 00:00:00
       2      1959-07-01 00:00:00
       3      1959-10-01 00:00:00
       4      1960-01-01 00:00:00
                     ...         
value  604                    6.0
       605                    6.9
       606                    8.1
       607                    9.2
       608                    9.6
Length: 1827, dtype: object

https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.pivot.html

#### you must specify index, columns and values in pivot

In [50]:
# if the data is in 'long' form, you can change it to 'wide' form with pivot
# the argument columns specifies the column to use to make the new columns, 
# and values specifies the column to use for populating the new values
# .pivot() is the inverse of melt()
# You can think of .pivot() as being similar to tidyverse's pivot_wider() function
ldata.pivot(index = 'date',columns = 'item',values = 'value').head()

item,infl,realgdp,unemp
date,,,
1959-01-01,0.00,2710.349,5.8
1959-04-01,2.34,2778.801,5.1
1959-07-01,2.74,2775.488,5.3
1959-10-01,0.27,2785.204,5.6
1960-01-01,2.31,2847.699,5.2


In [51]:
# if the data is in 'long' form, you can change it to 'wide' form with pivot
# in this example, we specify the index to be 'item', of which there are three
# the date values become the columns, of which there are 203
ldata.pivot(index = 'item',columns = 'date',values = 'value').head()

date,1959-01-01,1959-04-01,1959-07-01,1959-10-01,1960-01-01,1960-04-01,1960-07-01,1960-10-01,1961-01-01,1961-04-01,...,2007-04-01,2007-07-01,2007-10-01,2008-01-01,2008-04-01,2008-07-01,2008-10-01,2009-01-01,2009-04-01,2009-07-01
item,,,,,,,,,,,,,,,,,,,,,
infl,0.000,2.340,2.740,0.270,2.310,0.14,2.700,1.210,-0.400,1.470,...,2.750,3.450,6.380,2.820,8.530,-3.16,-8.79,0.94,3.370,3.560
realgdp,2710.349,2778.801,2775.488,2785.204,2847.699,2834.39,2839.022,2802.616,2819.264,2872.005,...,13203.977,13321.109,13391.249,13366.865,13415.266,13324.60,13141.92,12925.41,12901.504,12990.341
unemp,5.800,5.100,5.300,5.600,5.200,5.20,5.600,6.300,6.800,7.000,...,4.500,4.700,4.800,4.900,5.400,6.00,6.90,8.10,9.200,9.600


In [52]:
# all of these pivot operations return new objects and leave the original data unchanged
data.head()

item,realgdp,infl,unemp
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


# Pivot_table()

Pandas has another function called `pivot_table()`

`pivot()` is strictly for rearranging data, while `pivot_table()` is for rearranging **and** aggregating data.

`pivot_table()` will always take data in long form and make it wide. The argument `columns` specifies where the column names will come from.

In [53]:
# long-format data
df = pd.DataFrame({
    'Date': ['Jan 1', 'Jan 1', 'Jan 1', 'Jan 2'],
    'Fruit': ['Apple', 'Apple', 'Banana', 'Apple'],
    'Sold': [5, 3, 8, 6]
})

print(df)

    Date   Fruit  Sold
0  Jan 1   Apple     5
1  Jan 1   Apple     3
2  Jan 1  Banana     8
3  Jan 2   Apple     6


If we try to reshape this using `pivot()`, Pandas will hit the two Apple entries for Jan 1 and panic. 

It asks: "Do I put the 5 or the 3 in the intersection of 'Jan 1' and 'Apple'?" Because it doesn't know how to combine them, it throws an error.

In [54]:
# This will crash!
df.pivot(index='Date', columns='Fruit', values='Sold')

# Error: ValueError: Index contains duplicate entries, cannot reshape

ValueError: Index contains duplicate entries, cannot reshape

`pivot_table()` is built to handle this situation. When it finds duplicate index/column pairs, it applies an aggregation function to combine them.

By default, it calculates the mean, but for sales data, we usually want to add them together using `aggfunc='sum'`.

In [55]:
# pivot_table combines the 5 and the 3
wide_df = df.pivot_table(
    index='Date', 
    columns='Fruit', 
    values='Sold', 
    aggfunc='sum',     # Tell Pandas HOW to combine duplicates
    fill_value=0       # Clean up any missing (NaN) values with 0
)

print(wide_df)

Fruit  Apple  Banana
Date                
Jan 1      8       8
Jan 2      6       0


# Group By


In [56]:
pd.__version__

'3.0.1'

In [57]:
np.random.seed(1)
df = pd.DataFrame({'key1' : ['a', 'a', 'b', 'b', 'a'],
                   'key2' : ['one', 'two', 'one', 'two', 'one'],
                   'data1' : np.random.randint(20, size = 5),
                   'data2' : np.random.randint(20, size = 5)})
df

,key1,key2,data1,data2
0,a,one,5,11
1,a,two,11,5
2,b,one,12,15
3,b,two,8,0
4,a,one,9,16


In [58]:
grouped = df['data1'].groupby(df['key1'])
grouped

In [59]:
grouped.mean()

key1
a     8.333333
b    10.000000
Name: data1, dtype: float64

In [60]:
df

,key1,key2,data1,data2
0,a,one,5,11
1,a,two,11,5
2,b,one,12,15
3,b,two,8,0
4,a,one,9,16


### if there is a mix of numeric and categorical data, specify `numeric_only = True`

In [61]:
df.groupby(by = 'key1').mean(numeric_only = True)
# if you don't specify the column, it'll apply the function to the entire dataframe

,data1,data2
key1,,
a,8.333333,10.666667
b,10.000000,7.500000


In [62]:
df

,key1,key2,data1,data2
0,a,one,5,11
1,a,two,11,5
2,b,one,12,15
3,b,two,8,0
4,a,one,9,16


In [63]:
means = df['data1'].groupby([df['key1'], df['key2']]).mean()
means
# means has a multi-index

key1  key2
a     one      7.0
      two     11.0
b     one     12.0
      two      8.0
Name: data1, dtype: float64

In [64]:
# with the multi-index, you can unstack
means.unstack()

key2,one,two
key1,,
a,7.0,11.0
b,12.0,8.0


In [65]:
df

,key1,key2,data1,data2
0,a,one,5,11
1,a,two,11,5
2,b,one,12,15
3,b,two,8,0
4,a,one,9,16


In [66]:
# you can perform group by on Series that are not in the dataframe, but are of the correct length
states = np.array(['Ohio', 'California', 'California', 'Ohio', 'Ohio'])
years = np.array([2005, 2005, 2006, 2005, 2006])
df['data1'].groupby([states, years]).mean()

California  2005    11.0
            2006    12.0
Ohio        2005     6.5
            2006     9.0
Name: data1, dtype: float64

In [67]:
df

,key1,key2,data1,data2
0,a,one,5,11
1,a,two,11,5
2,b,one,12,15
3,b,two,8,0
4,a,one,9,16


In [68]:
df.groupby(['key1', 'key2']).size() # you don't always have to use mean, you can use other functions as well

key1  key2
a     one     2
      two     1
b     one     1
      two     1
dtype: int64

### Iterating over groups

In [69]:
df

,key1,key2,data1,data2
0,a,one,5,11
1,a,two,11,5
2,b,one,12,15
3,b,two,8,0
4,a,one,9,16


In [70]:
# the groupby creates a series of tuples that can be unpacked into name and group
for name, group in df.groupby('key1'):
    print("name:", name)
    print('------')
    print("group:\n", group)
    print('------')
    print("data1 mean:", group.data1.mean())
    print("data2 mean:", group.data2.mean())
    print('**************************')

name: a
------
group:
   key1 key2  data1  data2
0    a  one      5     11
1    a  two     11      5
4    a  one      9     16
------
data1 mean: 8.333333333333334
data2 mean: 10.666666666666666
**************************
name: b
------
group:
   key1 key2  data1  data2
2    b  one     12     15
3    b  two      8      0
------
data1 mean: 10.0
data2 mean: 7.5
**************************


In [71]:
for name, group in df.groupby('key2'):
    print("name:", name)
    print('------')
    print("group:\n", group)
    print('------')
    print("data1 mean:", group.data1.mean())
    print("data2 mean:", group.data2.mean())
    print('**************************')


name: one
------
group:
   key1 key2  data1  data2
0    a  one      5     11
2    b  one     12     15
4    a  one      9     16
------
data1 mean: 8.666666666666666
data2 mean: 14.0
**************************
name: two
------
group:
   key1 key2  data1  data2
1    a  two     11      5
3    b  two      8      0
------
data1 mean: 9.5
data2 mean: 2.5
**************************
